# 原代码加修改版

这个 notebook 保留原来的错误代码，并在错误代码下面追加修改后的正确代码，方便逐处对照。

**注意**：这个文件用于展示修改痕迹；如果要从头执行，请使用已经跑通的 `L3_Clustering_MonteCarlo.ipynb`。


# Lab 3: Clustering Standard Errors — Monte Carlo Study

**课程**：经济与商务实证研究方法（RMEB） 2026 Spring  
**主题**：聚类标准误的 Monte Carlo 验证  
**配套讲义**：Week 3 — 面板数据与经典 DiD

## 学习目标

- 在 Stata 中生成一个带序列相关的面板
- 复现 Bertrand-Duflo-Mullainathan (2004) 的 45% 过度拒绝结果
- 对比 iid / robust / 单聚类 / 两维聚类 / Wild Bootstrap 的推断表现
- 改变 AR(1) 系数 ρ 观察推断失真速度

## 参考文献

- Bertrand, M., Duflo, E., & Mullainathan, S. (2004). *QJE* 119(1), 249–275.
- Cameron, A. C., & Miller, D. L. (2015). *JHR* 50(2), 317–372.
- Cameron, A. C., Gelbach, J. B., & Miller, D. L. (2008). *ReStat* 90(3), 414–427.


## 1. 环境准备与包安装

运行前请确保已安装以下 Stata 外部命令（首次运行需要联网）：

## 包安装顺序

### 错误原代码（保留不变）


In [ ]:
clear all
set more off
set seed 20260516

* 安装必要包
capture ssc install reghdfe
capture ssc install ftools
capture ssc install boottest
capture ssc install estout

### 修改后正确代码

先安装 ftools，再安装 reghdfe。


In [ ]:
clear all
set more off
set seed 20260516

* 安装必要包
capture ssc install ftools
capture ssc install reghdfe
capture ssc install boottest
capture ssc install estout

## 2. 基本 DGP：500 单位 × 10 期面板

AR(1) 序列相关误差，随机选 100 个单位在 t=5 后接受"假政策"（真实效应为 0）。

## gen_panel 数据生成程序

### 错误原代码（保留不变）


In [ ]:
program define gen_panel
    syntax, [n(integer 500) t(integer 10) rho(real 0.8) te(real 0.0) seed(integer 0)]
    clear
    set seed `seed'
    set obs `n'
    gen unit = _n
    gen unit_fe = rnormal(0, 1)
    
    * 随机选 1/5 的单位作为处理组
    gen u = runiform()
    sort u
    gen treated = (_n <= `n' / 5)
    drop u
    
    expand `t'
    bysort unit: gen t = _n
    
    * 时间效应
    gen time_fe = rnormal(0, 0.5) if t == 1
    bysort t: egen tfe = mean(time_fe)
    drop time_fe
    rename tfe time_fe
    
    * AR(1) 误差：eps_it = rho * eps_{i,t-1} + u_it
    sort unit t
    gen eps = rnormal(0, 1) if t == 1
    replace eps = `rho' * L.eps + rnormal(0, 1) if t > 1 & !missing(L.eps)
    xtset unit t
    replace eps = `rho' * eps[_n-1] + rnormal(0, 1) if t > 1 & unit == unit[_n-1]
    
    * 处理指示与结果
    gen D = (treated == 1 & t >= 5)
    gen Y = unit_fe + time_fe + `te' * D + eps
end

### 修改后正确代码

修复重复定义、时间效应缺失和 AR(1) 误差生成顺序。


In [6]:
capture program drop gen_panel
program define gen_panel
    syntax, [n(integer 500) t(integer 10) rho(real 0.8) te(real 0.0) seed(integer 0)]
    clear
    set seed `seed'
    set obs `n'
    gen unit = _n
    gen unit_fe = rnormal(0, 1)
    
    * 随机选 1/5 的单位作为处理组
    gen u = runiform()
    sort u
    gen treated = (_n <= `n' / 5)
    drop u
    
    expand `t'
    bysort unit: gen t = _n
    
    * 时间效应
    sort t unit
    by t: gen time_fe = rnormal(0, 0.5) if _n == 1
    by t: replace time_fe = time_fe[1]
    
    * AR(1) 误差：eps_it = rho * eps_{i,t-1} + u_it
    xtset unit t
    gen eps = .
    by unit (t): replace eps = rnormal(0, 1) if _n == 1
    by unit (t): replace eps = `rho' * eps[_n-1] + rnormal(0, 1) if _n > 1
    
    * 处理指示与结果
    gen D = (treated == 1 & t >= 5)
    gen Y = unit_fe + time_fe + `te' * D + eps
end

## 3. 单次运行：观察三种标准误

In [2]:
gen_panel, n(500) t(10) rho(0.8) te(0.0) seed(42)

* iid 标准误
reghdfe Y D, absorb(unit t)
estimates store m_iid

* Robust (HC1)
reghdfe Y D, absorb(unit t) vce(robust)
estimates store m_hc

* 按单位聚类
reghdfe Y D, absorb(unit t) vce(cluster unit)
estimates store m_cl

esttab m_iid m_hc m_cl, star(* 0.10 ** 0.05 *** 0.01) ///
    stats(N r2) mtitles("iid" "HC1" "Cluster") b(3) se(3)


number of observations (_N) was 0, now 500
(4,500 observations created)
(4,990 missing values generated)
(4,990 real changes made)
       panel variable:  unit (strongly balanced)
        time variable:  t, 1 to 10
                delta:  1 unit
(5,000 missing values generated)
(500 real changes made)
(4,500 real changes made)

(MWFE estimator converged in 2 iterations)

HDFE Linear regression                            Number of obs   =      5,000
Absorbing 2 HDFE groups                           F(   1,   4490) =       0.08
                                                  Prob > F        =     0.7770
                                                  R-squared       =     0.6832
                                                  Adj R-squared   =     0.6473
                                                  Within R-sq.    =     0.0000
                                                  Root MSE        =     1.1299

-----------------------------------------------------------------------

## 4. Monte Carlo：300 次重复的拒绝率对比

这是 BDM(2004) 逻辑的完整复现。真实效应 = 0；若名义 5% 水平下拒绝率高于 5%，则推断失真。

In [3]:
postfile mcresults rep p_iid p_hc p_cl using mc_cluster.dta, replace

local nreps = 300
forvalues r = 1/`nreps' {
    quietly {
        gen_panel, n(500) t(10) rho(0.8) te(0.0) seed(`r')
        
        reghdfe Y D, absorb(unit t)
        local p_iid = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
        
        reghdfe Y D, absorb(unit t) vce(robust)
        local p_hc = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
        
        reghdfe Y D, absorb(unit t) vce(cluster unit)
        local p_cl = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
    }
    post mcresults (`r') (`p_iid') (`p_hc') (`p_cl')
    if mod(`r', 50) == 0 display "完成 `r' / `nreps'"
}
postclose mcresults

use mc_cluster.dta, clear
foreach se in iid hc cl {
    gen reject_`se' = (p_`se' < 0.05)
    quietly summarize reject_`se', meanonly
    display "标准误类型 `se': 拒绝率 = " %5.1f r(mean) * 100 "%"
}




完成 50 / 300
完成 100 / 300
完成 150 / 300
完成 200 / 300
完成 250 / 300
完成 300 / 300



标准误类型 iid: 拒绝率 =  28.7%
标准误类型 hc: 拒绝率 =  29.3%
标准误类型 cl: 拒绝率 =   6.3%


**期望结果**：

- iid 标准误：远高于 5%（可能 30-45%）
- HC1 稳健：仍明显高于 5%
- 按单位聚类：接近 5%

这就是为什么 Cameron-Miller 说：面板数据*必须聚类*。

## 5. 改变 ρ：观察推断退化速度

如果误差没有序列相关（ρ = 0），iid 标准误应当是正确的。随着 ρ 增加，推断失真加剧。

## rho 分组拒绝率输出

### 错误原代码（保留不变）


In [ ]:
* 跑三组 ρ 的 MC
postfile rho_results rho p_iid p_cl using mc_rho.dta, replace

foreach rho_val in 0.0 0.4 0.8 {
    forvalues r = 1/100 {
        quietly {
            gen_panel, n(500) t(10) rho(`rho_val') te(0.0) seed(`r')
            reghdfe Y D, absorb(unit t)
            local p_iid = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
            reghdfe Y D, absorb(unit t) vce(cluster unit)
            local p_cl = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
        }
        post rho_results (`rho_val') (`p_iid') (`p_cl')
    }
}
postclose rho_results

use mc_rho.dta, clear
gen reject_iid = (p_iid < 0.05)
gen reject_cl  = (p_cl  < 0.05)
table rho, stat(mean reject_iid reject_cl)

### 修改后正确代码

改用 Stata 16 兼容的 collapse + list。


In [4]:
* 跑三组 ρ 的 MC
postfile rho_results rho p_iid p_cl using mc_rho.dta, replace

foreach rho_val in 0.0 0.4 0.8 {
    forvalues r = 1/100 {
        quietly {
            gen_panel, n(500) t(10) rho(`rho_val') te(0.0) seed(`r')
            reghdfe Y D, absorb(unit t)
            local p_iid = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
            reghdfe Y D, absorb(unit t) vce(cluster unit)
            local p_cl = 2 * ttail(e(df_r), abs(_b[D] / _se[D]))
        }
        post rho_results (`rho_val') (`p_iid') (`p_cl')
    }
}
postclose rho_results

use mc_rho.dta, clear
gen reject_iid = (p_iid < 0.05)
gen reject_cl  = (p_cl  < 0.05)
collapse (mean) reject_iid reject_cl, by(rho)
format reject_iid reject_cl %6.3f
list rho reject_iid reject_cl, noobs











  +---------------------------+
  | rho   reject~d   reject~l |
  |---------------------------|
  |   0      0.040      0.030 |
  |  .4      0.140      0.030 |
  |  .8      0.200      0.050 |
  +---------------------------+


## 6. 小聚类数：Wild Bootstrap

当聚类数 G < 50 时，渐近推断不可靠。此时用 Wild Cluster Bootstrap（Cameron-Gelbach-Miller 2008）。

## Wild Cluster Bootstrap

### 错误原代码（保留不变）


In [ ]:
* 只保留前 30 个单位（模拟小 G 情境）
gen_panel, n(30) t(10) rho(0.8) te(0.0) seed(42)

reghdfe Y D, absorb(unit t) vce(cluster unit)

* Wild Cluster Bootstrap（999 次）
boottest D, boottype(wild-t) reps(999) cluster(unit)

### 修改后正确代码

改用显式单位和时间虚拟变量后再运行 boottest。


In [5]:
* 只保留前 30 个单位（模拟小 G 情境）
gen_panel, n(30) t(10) rho(0.8) te(0.0) seed(42)

reghdfe Y D, absorb(unit t) vce(cluster unit)

* Wild Cluster Bootstrap（999 次）
* boottest 在 Stata 16 中不能直接接在吸收多组固定效应的 reghdfe 后面；
* 小样本示例中改用显式单位和时间虚拟变量，结果与双向固定效应设定一致。
regress Y D i.unit i.t, vce(cluster unit)
boottest D, reps(999) cluster(unit) seed(20260516)


number of observations (_N) was 0, now 30
(270 observations created)
(290 missing values generated)
(290 real changes made)
       panel variable:  unit (strongly balanced)
        time variable:  t, 1 to 10
                delta:  1 unit
(300 missing values generated)
(30 real changes made)
(270 real changes made)

(MWFE estimator converged in 2 iterations)

HDFE Linear regression                            Number of obs   =        300
Absorbing 2 HDFE groups                           F(   1,     29) =       3.63
Statistics robust to heteroskedasticity           Prob > F        =     0.0668
                                                  R-squared       =     0.7138
                                                  Adj R-squared   =     0.6708
                                                  Within R-sq.    =     0.0261
Number of clusters (unit)    =         30         Root MSE        =     1.0442

                                  (Std. Err. adjusted for 30 clusters in unit)
----

## 7. 两维聚类

若误差既沿时间传染（行业波动）又沿空间传染（地区外溢），使用 Cameron-Gelbach-Miller (2011) 两维聚类。

In [ ]:
gen_panel, n(500) t(10) rho(0.8) te(0.0) seed(42)

* 假设每 25 个单位构成一个"行业"
gen industry = ceil(unit / 25)

reghdfe Y D, absorb(unit t) vce(cluster unit t)
reghdfe Y D, absorb(unit t) vce(cluster unit industry)

## 8. 练习

1. 把 `rho` 提到 0.95，把 `te` 保持为 0，跑 MC——观察失真程度。
2. 把 `n` 降到 30，用 Wild Bootstrap 与标准聚类做 MC 对比拒绝率差异。
3. 给定 `te = 0.3`（真实有效应），比较不同标准误下*功效*（power）的差异。
4. 用 Python 或 R 复现上述 MC，对比三种语言的速度。

## 9. 延伸阅读

- Abadie, Athey, Imbens, Wooldridge (2023). "When should you adjust standard errors for clustering?" *QJE* 138(1), 1–35.
- MacKinnon & Webb (2020). "Wild Bootstrap Randomization Inference for Few Treated Clusters." *JAE* 35(7), 888–908.